In [2]:
import numpy as np
import random
from collections import defaultdict

In [3]:
class TicTacToe:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [" "] * 9
        return self.get_state()

    def get_state(self):
        return "".join(self.board)

    def available_actions(self):
        return [i for i in range(9) if self.board[i] == " "]

    def make_move(self, action, player):
        if self.board[action] == " ":
            self.board[action] = player
            return True
        return False

    def check_winner(self):
        win_patterns = [
            (0, 1, 2), (3, 4, 5), (6, 7, 8),
            (0, 3, 6), (1, 4, 7), (2, 5, 8),
            (0, 4, 8), (2, 4, 6)
        ]

        for a, b, c in win_patterns:
            if self.board[a] == self.board[b] == self.board[c] != " ":
                return self.board[a]

        if " " not in self.board:
            return "Draw"

        return None

    def display(self):
        for i in range(0, 9, 3):
            print(" | ".join(self.board[i:i+3]))
            if i < 6:
                print("--+---+--")

In [4]:
def opponent_move(env):
    actions = env.available_actions()
    if actions:
        return random.choice(actions)
    return None

In [5]:
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=1.0):
        self.q_table = defaultdict(float)
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def choose_action(self, state, actions):
        if random.random() < self.epsilon:
            return random.choice(actions)

        q_values = [self.q_table[(state, a)] for a in actions]
        max_q = max(q_values)

        best_actions = [
            a for a, q in zip(actions, q_values) if q == max_q
        ]
        return random.choice(best_actions)

    def update(self, state, action, reward, next_state, next_actions, done):
        old_q = self.q_table[(state, action)]

        if done or not next_actions:
            future_q = 0
        else:
            future_q = max(
                self.q_table[(next_state, a)] for a in next_actions
            )

        self.q_table[(state, action)] = old_q + self.alpha * (
            reward + self.gamma * future_q - old_q
        )

In [6]:
env = TicTacToe()
agent = QLearningAgent()

episodes = 30000

for episode in range(episodes):
    env.reset()
    done = False

    while not done:
        state = env.get_state()
        actions = env.available_actions()
        action = agent.choose_action(state, actions)

        env.make_move(action, "X")
        result = env.check_winner()

        if result == "X":
            agent.update(state, action, 1, "", [], True)
            break

        if result == "Draw":
            agent.update(state, action, 0.2, "", [], True)
            break

        opponent_move(env)
        result = env.check_winner()

        if result == "O":
            agent.update(state, action, -1, "", [], True)
            break

        if result == "Draw":
            agent.update(state, action, 0.2, "", [], True)
            break

        next_state = env.get_state()
        next_actions = env.available_actions()

        agent.update(
            state, action, 0, next_state, next_actions, False
        )

    agent.epsilon = max(0.05, agent.epsilon * 0.9998)

print("Training completed!")
print("Total learned state-action pairs:", len(agent.q_table))

Training completed!
Total learned state-action pairs: 1297


In [7]:
def play_game(agent):
    env = TicTacToe()
    print("Cell numbering:")
    print("1 | 2 | 3")
    print("--+---+--")
    print("4 | 5 | 6")
    print("--+---+--")
    print("7 | 8 | 9")

    while True:
        state = env.get_state()
        action = agent.choose_action(state, env.available_actions())

        env.make_move(action, "X")
        print("\nAI's move:")
        env.display()

        result = env.check_winner()
        if result:
            break

        while True:
            try:
                user_move = int(input("Enter your move (1-9): ")) - 1
                if user_move in env.available_actions():
                    env.make_move(user_move, "O")
                    break
                print("Invalid move! Choose an empty cell.")
            except ValueError:
                print("Please enter a number from 1 to 9.")

        print("\nYour move:")
        env.display()

        result = env.check_winner()
        if result:
            break

    if result == "X":
        print("AI wins!")
    elif result == "O":
        print("You win!")
    else:
        print("It's a draw!")

# Disable exploration during testing
agent.epsilon = 0
play_game(agent)

Cell numbering:
1 | 2 | 3
--+---+--
4 | 5 | 6
--+---+--
7 | 8 | 9

AI's move:
  |   |  
--+---+--
  |   | X
--+---+--
  |   |  


Enter your move (1-9):  1



Your move:
O |   |  
--+---+--
  |   | X
--+---+--
  |   |  

AI's move:
O |   |  
--+---+--
  |   | X
--+---+--
  |   | X


Enter your move (1-9):  2



Your move:
O | O |  
--+---+--
  |   | X
--+---+--
  |   | X

AI's move:
O | O |  
--+---+--
  | X | X
--+---+--
  |   | X


Enter your move (1-9):  3



Your move:
O | O | O
--+---+--
  | X | X
--+---+--
  |   | X
You win!
